# Optimización de Modelos Conjunto Soleado por GMM

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('/content/drive/MyDrive/Tesina/03_Clusterizacion.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,7,77,0,4,15,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,7,82,0,4,16,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,9,85,0,3,16,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,11,87,0,3,16,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,11,88,0,3,16,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,15,86,0,3,14,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,47,89,0,3,16,6,Nublado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,51,95,0,4,17,7,Nublado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,47,100,0,3,18,8,Nublado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,5,100,1,4,18,9,Nublado,Lluvioso


In [5]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [6]:
datos_dia = datos.copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
24,2022-09-02 00:00:00,0.000000,19,6,76,0,4,15,0,Noche,Noche,0.000000,0.000000
25,2022-09-02 01:00:00,0.000000,18,7,81,0,4,15,1,Noche,Noche,0.000000,0.000000
26,2022-09-02 02:00:00,0.000000,18,7,84,0,4,15,2,Noche,Noche,0.000000,0.000000
27,2022-09-02 03:00:00,0.000000,18,7,86,0,4,15,3,Noche,Noche,0.000000,0.000000
28,2022-09-02 04:00:00,0.000000,17,7,86,0,4,15,4,Noche,Noche,0.000000,0.000000
29,2022-09-02 05:00:00,0.000000,17,7,88,0,4,15,5,Noche,Noche,0.000000,0.000000
30,2022-09-02 06:00:00,0.000000,17,7,91,0,4,15,6,Nublado,Lluvioso,0.000000,0.000000
31,2022-09-02 07:00:00,0.000000,17,7,94,0,3,16,7,Nublado,Lluvioso,0.000000,6.584959
32,2022-09-02 08:00:00,438.814997,16,5,97,0,3,15,8,Nublado,Lluvioso,0.000000,560.422022
33,2022-09-02 09:00:00,5908.000884,17,0,93,1,2,16,9,Nublado,Lluvioso,438.814997,7720.582326


In [7]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [8]:
X = datos_dia[columns]
X

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
24,19,6,76,0,4,15,0,0.0,0.0
25,18,7,81,0,4,15,1,0.0,0.0
26,18,7,84,0,4,15,2,0.0,0.0
27,18,7,86,0,4,15,3,0.0,0.0
28,17,7,86,0,4,15,4,0.0,0.0
...,...,...,...,...,...,...,...,...,...
18285,22,0,45,0,1,9,20,1450.0,0.0
18286,20,0,54,0,1,10,21,0.0,0.0
18287,18,0,62,0,1,11,22,0.0,0.0
18288,17,0,69,0,1,11,23,0.0,0.0


In [9]:
y = datos_dia[['Generación']]
y

,Generación
24,0.0
25,0.0
26,0.0
27,0.0
28,0.0
...,...
18285,0.0
18286,0.0
18287,0.0
18288,0.0


Dividimos entrenamiento, validación y prueba

In [10]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [11]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 12786, y_train: 12786
X_val: 2740, y_val: 2740
X_test: 2740, y_test: 2740


## Escalar con MinMaxScaler

In [12]:
from sklearn.preprocessing import MinMaxScaler

In [13]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [14]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[0.5        0.06666667 0.74736842 ... 0.         0.         0.        ]
 [0.47368421 0.07777778 0.8        ... 0.04347826 0.         0.        ]
 [0.47368421 0.07777778 0.83157895 ... 0.08695652 0.         0.        ]
 ...
 [0.36842105 0.68888889 0.77894737 ... 0.60869565 0.71913333 0.34156667]
 [0.36842105 0.54444444 0.75789474 ... 0.65217391 0.70556667 0.33793333]
 [0.36842105 0.65555556 0.72631579 ... 0.69565217 0.79483333 0.24856667]]
(12786, 9)


In [15]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
24,0.500000,0.066667,0.747368,0.000000,0.75,0.789474,0.000000,0.000000,0.000000
25,0.473684,0.077778,0.800000,0.000000,0.75,0.789474,0.043478,0.000000,0.000000
26,0.473684,0.077778,0.831579,0.000000,0.75,0.789474,0.086957,0.000000,0.000000
27,0.473684,0.077778,0.852632,0.000000,0.75,0.789474,0.130435,0.000000,0.000000
28,0.447368,0.077778,0.852632,0.000000,0.75,0.789474,0.173913,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
12805,0.342105,0.544444,0.736842,0.142857,1.00,0.473684,0.521739,0.843733,0.344767
12806,0.394737,0.544444,0.652632,0.142857,1.00,0.473684,0.565217,0.829000,0.338200
12807,0.368421,0.688889,0.778947,0.071429,1.00,0.526316,0.608696,0.719133,0.341567
12808,0.368421,0.544444,0.757895,0.071429,1.00,0.526316,0.652174,0.705567,0.337933


In [16]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.34210526 0.63333333 0.72631579 ... 0.73913043 0.70433333 0.24723333]
 [0.34210526 0.54444444 0.69473684 ... 0.7826087  0.702      0.2108    ]
 [0.34210526 0.48888889 0.69473684 ... 0.82608696 0.4138     0.01416667]
 ...
 [0.81578947 0.07777778 0.25263158 ... 0.7826087  0.9109     0.82416667]
 [0.78947368 0.06666667 0.30526316 ... 0.82608696 0.82416667 0.45596667]
 [0.73684211 0.05555556 0.35789474 ... 0.86956522 0.46683333 0.0394    ]]
(2740, 9)


In [17]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
12810,0.342105,0.633333,0.726316,0.000000,1.00,0.473684,0.739130,0.704333,0.247233
12811,0.342105,0.544444,0.694737,0.000000,1.00,0.421053,0.782609,0.702000,0.210800
12812,0.342105,0.488889,0.694737,0.000000,1.00,0.421053,0.826087,0.413800,0.014167
12813,0.315789,0.377778,0.726316,0.000000,0.75,0.368421,0.869565,0.043467,0.000000
12814,0.289474,0.377778,0.768421,0.000000,0.50,0.368421,0.913043,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
15545,0.894737,0.077778,0.178947,0.357143,0.00,0.473684,0.695652,0.959467,0.935400
15546,0.868421,0.077778,0.200000,0.214286,0.25,0.473684,0.739130,0.940767,0.910900
15547,0.815789,0.077778,0.252632,0.142857,0.50,0.578947,0.782609,0.910900,0.824167
15548,0.789474,0.066667,0.305263,0.071429,0.25,0.631579,0.826087,0.824167,0.455967


In [18]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.68421053 0.05555556 0.42105263 ... 0.91304348 0.0379     0.        ]
 [0.65789474 0.05555556 0.49473684 ... 0.95652174 0.         0.        ]
 [0.60526316 0.05555556 0.56842105 ... 1.         0.         0.        ]
 ...
 [0.47368421 0.         0.6        ... 0.95652174 0.         0.        ]
 [0.44736842 0.         0.67368421 ... 1.         0.         0.        ]
 [0.42105263 0.         0.71578947 ... 0.         0.         0.        ]]
(2740, 9)


In [19]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
15550,0.684211,0.055556,0.421053,0.0,0.0,0.736842,0.913043,0.037900,0.0
15551,0.657895,0.055556,0.494737,0.0,0.0,0.736842,0.956522,0.000000,0.0
15552,0.605263,0.055556,0.568421,0.0,0.0,0.789474,1.000000,0.000000,0.0
15553,0.578947,0.055556,0.600000,0.0,0.0,0.789474,0.000000,0.000000,0.0
15554,0.578947,0.044444,0.631579,0.0,0.0,0.789474,0.043478,0.000000,0.0
...,...,...,...,...,...,...,...,...,...
18285,0.578947,0.000000,0.421053,0.0,0.0,0.473684,0.869565,0.048333,0.0
18286,0.526316,0.000000,0.515789,0.0,0.0,0.526316,0.913043,0.000000,0.0
18287,0.473684,0.000000,0.600000,0.0,0.0,0.578947,0.956522,0.000000,0.0
18288,0.447368,0.000000,0.673684,0.0,0.0,0.578947,1.000000,0.000000,0.0


In [20]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [21]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[0.48717949 0.06666667 0.75257732 ... 0.         0.         0.        ]
 [0.46153846 0.07777778 0.80412371 ... 0.04347826 0.         0.        ]
 [0.46153846 0.07777778 0.83505155 ... 0.08695652 0.         0.        ]
 ...
 [0.46153846 0.         0.60824742 ... 0.95652174 0.         0.        ]
 [0.43589744 0.         0.68041237 ... 1.         0.         0.        ]
 [0.41025641 0.         0.72164948 ... 0.         0.         0.        ]]
(18266, 9)


In [22]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
24,0.487179,0.066667,0.752577,0.0,0.75,0.75,0.000000,0.000000,0.0
25,0.461538,0.077778,0.804124,0.0,0.75,0.75,0.043478,0.000000,0.0
26,0.461538,0.077778,0.835052,0.0,0.75,0.75,0.086957,0.000000,0.0
27,0.461538,0.077778,0.855670,0.0,0.75,0.75,0.130435,0.000000,0.0
28,0.435897,0.077778,0.855670,0.0,0.75,0.75,0.173913,0.000000,0.0
...,...,...,...,...,...,...,...,...,...
18285,0.564103,0.000000,0.432990,0.0,0.00,0.45,0.869565,0.048333,0.0
18286,0.512821,0.000000,0.525773,0.0,0.00,0.50,0.913043,0.000000,0.0
18287,0.461538,0.000000,0.608247,0.0,0.00,0.55,0.956522,0.000000,0.0
18288,0.435897,0.000000,0.680412,0.0,0.00,0.55,1.000000,0.000000,0.0


In [23]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [24]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.        ]
 [0.        ]
 [0.        ]
 ...
 [0.70556667]
 [0.79483333]
 [0.70433333]]
(12786, 1)


In [25]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
24,0.000000
25,0.000000
26,0.000000
27,0.000000
28,0.000000
...,...
12805,0.829000
12806,0.719133
12807,0.705567
12808,0.794833


In [26]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[0.702     ]
 [0.4138    ]
 [0.04346667]
 ...
 [0.82416667]
 [0.46683333]
 [0.0379    ]]
(2740, 1)


In [27]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
12810,0.702000
12811,0.413800
12812,0.043467
12813,0.000000
12814,0.000000
...,...
15545,0.940767
15546,0.910900
15547,0.824167
15548,0.466833


In [28]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[0.]
 [0.]
 [0.]
 ...
 [0.]
 [0.]
 [0.]]
(2740, 1)


In [29]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
15550,0.0
15551,0.0
15552,0.0
15553,0.0
15554,0.0
...,...
18285,0.0
18286,0.0
18287,0.0
18288,0.0


In [30]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [31]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.]
 [0.]
 [0.]
 ...
 [0.]
 [0.]
 [0.]]
(18266, 1)


In [32]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
24,0.0
25,0.0
26,0.0
27,0.0
28,0.0
...,...
18285,0.0
18286,0.0
18287,0.0
18288,0.0


## Preparación para Redes Neuronales

In [33]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []

    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada

        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [34]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [35]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (12738, 48, 9), y_train: (12738, 1)
X_val: (2692, 48, 9), y_val: (2692, 1)
X_test: (2692, 48, 9), y_test: (2692, 1)


## Optuna

In [36]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.8/231.8 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 8.2 MB/s eta 0:00:00


In [37]:
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

### Optimización de LightGBM

In [38]:
import optuna
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

# Definir la función de optimización
def objective(trial):
    # Espacio de búsqueda para los hiperparámetros
    num_leaves = trial.suggest_int('num_leaves', 10, 1000)
    subsample = trial.suggest_float('subsample', 0.1, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.1, 1.0)
    min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 10, 100)

    # Modelo LightGBM con los hiperparámetros sugeridos
    model = LGBMRegressor(
        num_leaves=num_leaves,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_data_in_leaf=min_data_in_leaf,
        random_state=42
    )

    # Entrenamos el modelo con los datos de entrenamiento
    model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Ejecutar la optimización con 50 iteraciones
LightGBM_study = optuna.create_study(direction="minimize")
LightGBM_study.optimize(objective, n_trials=50)

# Obtener los mejores hiperparámetros
best_params = LightGBM_study.best_params
print("Mejores hiperparámetros:", best_params)

[I 2025-03-14 03:39:02,855] A new study created in memory with name: no-name-29483e17-76b0-474c-8cd9-d6b3447bc7b0


[LightGBM] [Warning] min_data_in_leaf is set=12, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=12
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=12, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=12
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001074 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 775
[LightGBM] [Info] Number of data points in the train set: 12786, number of used features: 9
[LightGBM] [Info] Start training from score 0.296336
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with pos

[I 2025-03-14 03:39:03,965] Trial 0 finished with value: 0.004460320061011488 and parameters: {'num_leaves': 586, 'subsample': 0.8703575195866381, 'colsample_bytree': 0.9342631189015203, 'min_data_in_leaf': 12}. Best is trial 0 with value: 0.004460320061011488.
[I 2025-03-14 03:39:04,023] Trial 1 finished with value: 0.00383516470056821 and parameters: {'num_leaves': 10, 'subsample': 0.4689678354498932, 'colsample_bytree': 0.18130343183883546, 'min_data_in_leaf': 89}. Best is trial 1 with value: 0.00383516470056821.


[LightGBM] [Warning] min_data_in_leaf is set=12, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=12
[LightGBM] [Warning] min_data_in_leaf is set=89, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=89
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=89, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=89
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000072 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 775
[LightGBM] [Info] Number of data points in the train set: 12786, number of used features: 9
[LightGBM] [Info] Start training from score 0.296336
[LightGBM] [Warning] min_data_in_leaf is set=89, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=89
[LightGBM] [Warning] min_dat

[I 2025-03-14 03:39:04,408] Trial 2 finished with value: 0.0037709594307690224 and parameters: {'num_leaves': 823, 'subsample': 0.3693875873764938, 'colsample_bytree': 0.2899865315820669, 'min_data_in_leaf': 30}. Best is trial 2 with value: 0.0037709594307690224.


...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-14 03:39:13,051] Trial 48 finished with value: 0.0034902452497675607 and parameters: {'num_leaves': 154, 'subsample': 0.346957767253953, 'colsample_bytree': 0.3214758808584802, 'min_data_in_leaf': 85}. Best is trial 47 with value: 0.003302467360722824.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2025-03-14 03:39:13,124] Trial 49 finished with value: 0.003390387695555164 and parameters: {'num_leaves': 15, 'subsample': 0.24678696012201373, 'colsample_bytree': 0.4224343550627956, 'min_data_in_leaf': 96}. Best is trial 47 with value: 0.003302467360722824.


[LightGBM] [Warning] min_data_in_leaf is set=96, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=96
Mejores hiperparámetros: {'num_leaves': 14, 'subsample': 0.3412949971354309, 'colsample_bytree': 0.34190269422505076, 'min_data_in_leaf': 85}


### Random Forest

In [39]:
def objective(trial):
    # Definir los hiperparámetros a optimizar
    n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)
    max_depth = trial.suggest_int("max_depth", 10, 50, step=5)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    # Definir el modelo con los hiperparámetros actuales
    RF_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        criterion="squared_error",
        random_state=0
    )

    # Entrenar el modelo
    RF_model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = RF_model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Crear el estudio de optimización
RF_study = optuna.create_study(direction="minimize")
RF_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", RF_study.best_params)

[I 2025-03-14 03:39:13,131] A new study created in memory with name: no-name-16efcb17-b32a-4603-9e1b-b302e1785df4
[I 2025-03-14 03:39:18,151] Trial 0 finished with value: 0.005188570311313326 and parameters: {'n_estimators': 100, 'max_depth': 15, 'min_samples_split': 17, 'min_samples_leaf': 6, 'bootstrap': False}. Best is trial 0 with value: 0.005188570311313326.
[I 2025-03-14 03:39:25,619] Trial 1 finished with value: 0.0040518477406828405 and parameters: {'n_estimators': 200, 'max_depth': 20, 'min_samples_split': 7, 'min_samples_leaf': 4, 'bootstrap': True}. Best is trial 1 with value: 0.0040518477406828405.
[I 2025-03-14 03:39:48,722] Trial 2 finished with value: 0.004116891441970541 and parameters: {'n_estimators': 500, 'max_depth': 25, 'min_samples_split': 4, 'min_samples_leaf': 1, 'bootstrap': True}. Best is trial 1 with value: 0.0040518477406828405.
[I 2025-03-14 03:39:58,659] Trial 3 finished with value: 0.005065759894801997 and parameters: {'n_estimators': 250, 'max_depth': 10

Mejores hiperparámetros: {'n_estimators': 400, 'max_depth': 50, 'min_samples_split': 4, 'min_samples_leaf': 10, 'bootstrap': True}


### CTNET

In [40]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])

    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

# Construir modelo con hiperparámetros sugeridos
def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout, mlp_dropout):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs

    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)

    for units in mlp_units:
        x = layers.Dense(units, activation="relu")(x)

    x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(1)(x)
    return tf.keras.Model(inputs, outputs)

In [41]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [42]:
# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    head_size = trial.suggest_int("head_size", 2, 8)
    num_heads = trial.suggest_int("num_heads", 2, 8)
    ff_dim = trial.suggest_int("ff_dim", 16, 128, step=16)
    num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 5)
    mlp_units = [trial.suggest_int("mlp_units_1", 64, 512, step=64),
                 trial.suggest_int("mlp_units_2", 32, 256, step=32)]
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Construir el modelo
    model = build_model(
        input_shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]),
        head_size=head_size,
        num_heads=num_heads,
        ff_dim=ff_dim,
        num_transformer_blocks=num_transformer_blocks,
        mlp_units=mlp_units,
        dropout=dropout,
        mlp_dropout=mlp_dropout
    )

    # Compilar y entrenar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Minimizar MSE

# Crear el estudio de optimización
CTNET_study = optuna.create_study(direction="minimize")
CTNET_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", CTNET_study.best_params)


[I 2025-03-14 03:50:06,650] A new study created in memory with name: no-name-57d72e1c-8d08-418e-9488-fa4588a145be
[I 2025-03-14 03:51:40,808] Trial 0 finished with value: 0.00515621667727828 and parameters: {'head_size': 8, 'num_heads': 8, 'ff_dim': 64, 'num_transformer_blocks': 2, 'mlp_units_1': 512, 'mlp_units_2': 160, 'dropout': 0.15681199539014357, 'mlp_dropout': 0.24248167102122142, 'learning_rate': 7.753923892396329e-05, 'batch_size': 128}. Best is trial 0 with value: 0.00515621667727828.
[I 2025-03-14 03:53:00,530] Trial 1 finished with value: 0.008374189026653767 and parameters: {'head_size': 5, 'num_heads': 7, 'ff_dim': 112, 'num_transformer_blocks': 3, 'mlp_units_1': 320, 'mlp_units_2': 192, 'dropout': 0.1959299692043972, 'mlp_dropout': 0.42683576953510927, 'learning_rate': 0.0009987655888165038, 'batch_size': 512}. Best is trial 0 with value: 0.00515621667727828.
[I 2025-03-14 03:54:29,377] Trial 2 finished with value: 0.007011353969573975 and parameters: {'head_size': 2, 'n

Mejores hiperparámetros: {'head_size': 2, 'num_heads': 7, 'ff_dim': 80, 'num_transformer_blocks': 1, 'mlp_units_1': 192, 'mlp_units_2': 128, 'dropout': 0.4987492446198915, 'mlp_dropout': 0.2048079774950788, 'learning_rate': 0.004694195402090184, 'batch_size': 128}


### Forescasting

In [43]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Conv1D, BatchNormalization, MaxPooling1D, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters = trial.suggest_categorical("filters", [32, 64, 128])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    lstm_units_1 = trial.suggest_categorical("lstm_units_1", [64, 128, 256])
    lstm_units_2 = trial.suggest_categorical("lstm_units_2", [32, 64, 128])
    lstm_units_3 = trial.suggest_categorical("lstm_units_3", [16, 32, 64])
    dropout_lstm = trial.suggest_float("dropout_lstm", 0.1, 0.5)
    dropout_dense = trial.suggest_float("dropout_dense", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])  # 🔹 Optimización del batch size

    # Construcción del modelo
    model = Sequential()
    model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

    # CNN
    model.add(Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))

    # BiLSTM
    model.add(Bidirectional(LSTM(lstm_units_1, return_sequences=True)))
    model.add(Bidirectional(LSTM(lstm_units_2, return_sequences=True)))
    model.add(Dropout(dropout_lstm))
    model.add(Bidirectional(LSTM(lstm_units_3, return_sequences=False)))

    # Normalización y Dropout
    model.add(BatchNormalization())
    model.add(Dropout(dropout_dense))

    # Capas Densas
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='relu'))

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Entrenar el modelo con validación
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Forecasting_study = optuna.create_study(direction="minimize")
Forecasting_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Forecasting_study.best_params)

[I 2025-03-14 04:40:28,804] A new study created in memory with name: no-name-c02af535-8173-438e-9c42-63e490ce88ad
[I 2025-03-14 04:41:57,238] Trial 9 finished with value: 0.2601148188114166 and parameters: {'filters': 32, 'kernel_size': 5, 'lstm_units_1': 256, 'lstm_units_2': 32, 'lstm_units_3': 32, 'dropout_lstm': 0.15204021684371316, 'dropout_dense': 0.394857929527083, 'learning_rate': 0.00016889975252812306, 'batch_size': 512}. Best is trial 9 with value: 0.2601148188114166.
[I 2025-03-14 04:42:21,606] Trial 12 finished with value: 0.2893924415111542 and parameters: {'filters': 32, 'kernel_size': 4, 'lstm_units_1': 256, 'lstm_units_2': 128, 'lstm_units_3': 32, 'dropout_lstm': 0.27127128163565273, 'dropout_dense': 0.4190494745458123, 'learning_rate': 0.0003954628034253596, 'batch_size': 128}. Best is trial 9 with value: 0.2601148188114166.
[I 2025-03-14 04:42:43,404] Trial 13 finished with value: 0.2694973647594452 and parameters: {'filters': 128, 'kernel_size': 4, 'lstm_units_1': 25

Mejores hiperparámetros: {'filters': 128, 'kernel_size': 5, 'lstm_units_1': 64, 'lstm_units_2': 128, 'lstm_units_3': 64, 'dropout_lstm': 0.40086998693335835, 'dropout_dense': 0.41776351065139583, 'learning_rate': 0.0012316082606971734, 'batch_size': 128}


### Photovoltaic

In [44]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Bidirectional, GRU, MultiHeadAttention
from tensorflow.keras.layers import Flatten, Dropout, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters_1 = trial.suggest_categorical("filters_1", [32, 64, 128])
    filters_2 = trial.suggest_categorical("filters_2", [64, 128, 256])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    gru_units = trial.suggest_categorical("gru_units", [32, 64, 128])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8])
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    dense_units_1 = trial.suggest_categorical("dense_units_1", [32, 64, 128])
    dense_units_2 = trial.suggest_categorical("dense_units_2", [16, 32, 64])
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Definir el modelo
    inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

    # Primera capa CNN
    x = Conv1D(filters=filters_1, kernel_size=kernel_size, padding='same', activation='relu')(inputs)
    x = MaxPooling1D(pool_size=2)(x)

    # Segunda capa CNN
    x = Conv1D(filters=filters_2, kernel_size=kernel_size, padding='same', activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)

    # Capa BiGRU
    x = Bidirectional(GRU(gru_units, return_sequences=True))(x)

    # Atención MultiHead
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=128)(x, x)

    # Aplanar y Dropout
    x = Flatten()(attention)
    x = Dropout(dropout_rate)(x)
    x = Dense(dense_units_1, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = Dense(dense_units_2, activation="relu")(x)

    # Capa de salida
    outputs = Dense(1, activation="linear")(x)

    # Construcción del modelo
    model = Model(inputs=inputs, outputs=outputs)

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Early Stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

    # Entrenar el modelo
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se detendrá con Early Stopping
        batch_size=batch_size,
        validation_data=(X_val_windowed, y_val_windowed),
        callbacks=[early_stop],
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Photovoltaic_study = optuna.create_study(direction="minimize")
Photovoltaic_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Photovoltaic_study.best_params)

[I 2025-03-14 05:04:15,787] A new study created in memory with name: no-name-f35aa84e-c62a-44c6-bbaf-2e42df2e01fa


Epoch 32: early stopping
Restoring model weights from the end of the best epoch: 22.


[I 2025-03-14 05:06:37,853] Trial 0 finished with value: 0.004994676448404789 and parameters: {'filters_1': 128, 'filters_2': 256, 'kernel_size': 3, 'gru_units': 128, 'num_heads': 4, 'dropout_rate': 0.2309712393454935, 'dense_units_1': 64, 'dense_units_2': 32, 'learning_rate': 0.0008633799216018332, 'batch_size': 512}. Best is trial 0 with value: 0.004994676448404789.


Epoch 35: early stopping
Restoring model weights from the end of the best epoch: 25.


[I 2025-03-14 05:06:43,833] Trial 4 finished with value: 0.005220108665525913 and parameters: {'filters_1': 32, 'filters_2': 64, 'kernel_size': 5, 'gru_units': 32, 'num_heads': 2, 'dropout_rate': 0.21200305336684744, 'dense_units_1': 32, 'dense_units_2': 64, 'learning_rate': 0.0056132676957964854, 'batch_size': 512}. Best is trial 0 with value: 0.004994676448404789.


...

Epoch 25: early stopping
Restoring model weights from the end of the best epoch: 15.


[I 2025-03-14 05:22:12,139] Trial 47 finished with value: 0.004949990194290876 and parameters: {'filters_1': 128, 'filters_2': 128, 'kernel_size': 3, 'gru_units': 64, 'num_heads': 2, 'dropout_rate': 0.29530333379157525, 'dense_units_1': 128, 'dense_units_2': 32, 'learning_rate': 0.00018862833760314781, 'batch_size': 128}. Best is trial 35 with value: 0.00454827630892396.


Epoch 27: early stopping
Restoring model weights from the end of the best epoch: 17.


[I 2025-03-14 05:22:14,217] Trial 46 finished with value: 0.004902360960841179 and parameters: {'filters_1': 128, 'filters_2': 128, 'kernel_size': 3, 'gru_units': 64, 'num_heads': 2, 'dropout_rate': 0.3929749428401735, 'dense_units_1': 128, 'dense_units_2': 32, 'learning_rate': 0.0002294997410343165, 'batch_size': 128}. Best is trial 35 with value: 0.00454827630892396.


Mejores hiperparámetros: {'filters_1': 128, 'filters_2': 128, 'kernel_size': 3, 'gru_units': 64, 'num_heads': 2, 'dropout_rate': 0.36428579949557205, 'dense_units_1': 128, 'dense_units_2': 32, 'learning_rate': 0.00029394651991630923, 'batch_size': 128}
